# Supplementary Table 5

Individual-function Spearman correlations between microbial DNA, RNA and protein layers across 77 PREDICT patients: 30,581 function-by-layer-pair tests in nine Benjamini-Hochberg families (three functional representations by three layer pairs), the nine-family summary, the eligibility of every candidate function, and the comparison with the previous functional allocation. This notebook rebuilds the workbook from `source_data/STab5/`, compares it cell by cell with the table as published, recomputes every test from the patient-level profiles and prints the numbers quoted in the text.

Run from the repository root: `python paper.py build STab5`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

Five sheets: `Read_me`, `All_30581_tests` (one row per eligible function and layer pair: patients, Spearman rho, two-sided P, Benjamini-Hochberg q within the family), `Nine_family_summary`, `All_eligibility` (184,386 candidate function-by-layer-pair rows) and `Eligibility_common` (functions eligible under both the previous and the revised allocation). Supports "Supplementary Table 5 | Individual-function correlations between DNA, RNA and protein layers." (v28d line 483) and the sentences on v28d lines 141 and 829. n = 77 patients; unit of each test: patient. The workbook is written by the producer's code and then edited as in the Source Data package (Read_me sheet replaced, `Eligibility_common` columns renamed). Its cell values must equal `expected/Supplementary_Table_5.xlsx`; the two support files must be byte-identical to the package copies.

In [ ]:
import hashlib, json
import pandas as pd
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
d = panels.load(SD)
xlsx, matched, ver = panels.build_table(SD, OUT)
elig_bytes, summary_bytes = panels.package_support(matched, ver)
(OUT / "eligibility_matched_correlations.tsv").write_bytes(elig_bytes)
(OUT / "common_eligibility_summary.tsv").write_bytes(summary_bytes)
exp = UNIT / "expected"
table_receipt = {
    "xlsx": panels.compare_xlsx_values(xlsx, exp / "Supplementary_Table_5.xlsx"),
    "eligibility_matched_correlations.tsv": {"new_sha256": sha(OUT / "eligibility_matched_correlations.tsv"),
                                             "expected_sha256": sha(exp / "eligibility_matched_correlations.tsv")},
    "common_eligibility_summary.tsv": {"new_sha256": sha(OUT / "common_eligibility_summary.tsv"),
                                       "expected_sha256": sha(exp / "common_eligibility_summary.tsv")},
}
table_receipt["xlsx"]["expected_sha256"] = sha(exp / "Supplementary_Table_5.xlsx")
for k in ("eligibility_matched_correlations.tsv", "common_eligibility_summary.tsv"):
    table_receipt[k]["identical"] = table_receipt[k]["new_sha256"] == table_receipt[k]["expected_sha256"]
table_receipt["result"] = "IDENTICAL" if all(v["identical"] for v in table_receipt.values()) else "DIFFERENT"
print(json.dumps({k: (v if k == "result" else v["identical"]) for k, v in table_receipt.items()}, indent=1))
print("cells compared:", table_receipt["xlsx"]["cells"], "exactly equal:", table_receipt["xlsx"]["exact_cells"])
pd.read_csv(OUT / "common_eligibility_summary.tsv", sep="\t")

## Recomputation of the 30,581 tests

For each representation (OG, KO, pathway) and each pair of layers, profiles of the 77 patients are normalized to relative signal over the function identifiers shared by that pair; each function with positive values in both layers in at least 35 patients gets one Spearman correlation (two-sided P, `scipy.stats.spearmanr`); P values are adjusted by Benjamini-Hochberg within each of the nine families (Methods, v28d line 829). The recomputed tests, eligibility table and summary are compared with the tables as produced (`source_data/STab5/`), and the recomputed test table is written in the producer's format and compared byte for byte.

In [ ]:
tests, eligibility, summary = panels.stat_individual_functions(SD)
K = ["level", "layer_a", "layer_b", "feature"]
recompute_receipt = {
    "tests": panels.compare_frames(tests, d["tests"], K, ["rho", "p", "q"], ["n_patients"]),
    "eligibility": panels.compare_frames(eligibility, d["eligibility"], K, [], ["n_patients", "eligible"]),
    "summary": panels.compare_frames(summary, d["summary"], K[:3], ["median_rho"], ["features", "significant"]),
}
tests.to_csv(OUT / "recomputed_tests.tsv", sep="\t", index=False)
recompute_receipt["tests_tsv_byte_identical"] = sha(OUT / "recomputed_tests.tsv") == sha(SD / "crossomics_same_function_correlations.tsv")
ok = (all(v["keys_equal"] and all(x for k, x in v.items() if k.startswith("equal_")) for k, v in recompute_receipt.items() if isinstance(v, dict))
      and all(x <= 1e-12 for v in recompute_receipt.values() if isinstance(v, dict) for k, x in v.items() if k.startswith("max_abs_diff")))
recompute_receipt["result"] = "MATCH" if ok else "DIFFERENT"
print(json.dumps(recompute_receipt, indent=1))
summary

## Statistics

One row per Benjamini-Hochberg family: family size, patients per test, median and interquartile range of rho, tests with q < 0.05, smallest P and q. These fill `STATS.tsv`.

In [ ]:
families = panels.stat_families(d["tests"])
pd.set_option("display.precision", 17)
families

## Checks against the table as produced and recomputed numbers

Writes the receipts `checks/table.json` and `checks/recompute.json`, then prints every number in `NUMBERS.tsv` order at full precision. There are no figure panels in this unit, so there are no pixel checks.

In [ ]:
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "table.json").write_text(json.dumps(table_receipt, indent=1) + "\n")
(UNIT / "checks" / "recompute.json").write_text(json.dumps(recompute_receipt, indent=1) + "\n")
print("table:", table_receipt["result"], "| recompute:", recompute_receipt["result"])
assert table_receipt["result"] == "IDENTICAL" and recompute_receipt["result"] == "MATCH"
for label, value in panels.numbers(summary, ver, tests, d["previous"]):
    print(f"{label}\t{value!r}")